In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from typing import Union
import warnings
warnings.filterwarnings('ignore')
from DATA.stock_invest_function import *

def fetch_valuation_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = 'US_company_valuation_result') -> Union[pd.DataFrame, None]:


    """
    지정한 ticker의 밸류에이션 결과를 조회하여 DataFrame으로 반환.
    결과가 없으면 메시지 출력 후 None 반환.
    """
    # 안전을 위해 입력 정규화 (대문자화/공백제거)
    q_ticker = (ticker or "").strip().upper()
    if not q_ticker:
        print("ticker 값을 올바르게 입력해 주세요.")
        return None

    # SQLAlchemy 엔진 생성
    connection_string = (
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )
    engine = create_engine(connection_string)

    try:
        # 테이블 존재 여부(옵션): 없으면 깔끔히 리턴
        with engine.connect() as conn:
            # MariaDB/ MySQL에서 information_schema로 존재 체크
            exists_sql = text("""
                SELECT COUNT(*) AS cnt
                FROM information_schema.tables
                WHERE table_schema = :schema AND table_name = :table
            """)
            exists = conn.execute(exists_sql, {"schema": db_info['database'], "table": table_name}).scalar()
            if not exists:
                print(f"테이블 '{table_name}' 이(가) 존재하지 않습니다.")
                return None

            # 본 조회 (ticker 컬럼 기준; 필요한 경우 인덱스 사용 권장)
            query = text(f"""
                SELECT *
                FROM {table_name}
                WHERE UPPER(ticker) = :ticker
                ORDER BY 1
            """)
            df = pd.read_sql(query, conn, params={"ticker": q_ticker})

        if df.empty:
            print("조회한  ticker의 기업 valation이 존재하지 않습니다")
            return None

        # datetime 컬럼 자동 파싱이 필요하면 여기서 처리 가능
        # for col in df.columns:
        #     if col.lower().endswith(('date', 'dt', 'time', 'timestamp')):
        #         df[col] = pd.to_datetime(df[col], errors='ignore')

        return df

    except Exception as e:
        print(f"조회 중 오류가 발생했습니다: {e}")
        return None
    finally:
        engine.dispose()

In [32]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

df = fetch_valuation_by_ticker(db_info, "PANW")
# if df is not None:
#     print(df.head())   # 앞부분 출력
print("=== 데이터 기본 정보 ===")
print(f"데이터 형태: {df.shape}")
print(f"컬럼: {list(df.columns)}")

# 각 indicator별 데이터 개수 확인
print("\n=== 각 indicator별 데이터 개수 ===")
indicator_counts = df['indicator'].value_counts()
print(indicator_counts)

# 1. revenue_with_exog, revenue_without_exog 데이터 합산
# forward_revenue, revenue 모두 포함
revenue_with_exog_sum = (
    df[df['indicator'] == 'forward_revenue_with_exog']['value'].sum() +
    df[df['indicator'] == 'revenue_with_exog']['value'].sum()
)

revenue_without_exog_sum = (
    df[df['indicator'] == 'forward_revenue_without_exog']['value'].sum() +
    df[df['indicator'] == 'revenue_without_exog']['value'].sum()
)

revenue_with_exog_sum = (
    df[df['indicator'] == 'revenue_with_exog']['value'].sum()
)

revenue_without_exog_sum = (
    df[df['indicator'] == 'revenue_without_exog']['value'].sum()
)

print("\n=== 매출 합산 결과 ===")
print(f"향후 4분기 예상매출_외생반영: {revenue_with_exog_sum:.2f}")
print(f"향후 4분기 예상매출_외생미반영: {revenue_without_exog_sum:.2f}")

# 2. Value 관련 데이터 추출 및 평균 계산

# value_with_exog - 실제 컬럼명 사용, 값에 4를 곱한 후 평균 계산
value_with_exog_data = df[df['indicator'] == 'value_with_exog']['value'] * 4
value_with_exog_avg = value_with_exog_data.mean() if not value_with_exog_data.empty else 0

# value_without_exog - 실제 컬럼명 사용, 값에 4를 곱한 후 평균 계산
value_without_exog_data = df[df['indicator'] == 'value_without_exog']['value'] * 4
value_without_exog_avg = value_without_exog_data.mean() if not value_without_exog_data.empty else 0

# lstm_value_with_exog - 실제 컬럼명 사용
lstm_with_exog_data = df[df['indicator'] == 'lstm_value_with_exog']['value']
lstm_with_exog_avg = lstm_with_exog_data.mean() if not lstm_with_exog_data.empty else 0

# lstm_value_without_exog - 실제 컬럼명 사용
lstm_without_exog_data = df[df['indicator'] == 'lstm_value_without_exog']['value']
lstm_without_exog_avg = lstm_without_exog_data.mean() if not lstm_without_exog_data.empty else 0

# prophet_value_with_exog - 실제 컬럼명 사용
prophet_with_exog_data = df[df['indicator'] == 'prophet_value_with_exog']['value']
prophet_with_exog_avg = prophet_with_exog_data.mean() if not prophet_with_exog_data.empty else 0

# prophet_value_without_exog - 실제 컬럼명 사용
prophet_without_exog_data = df[df['indicator'] == 'prophet_value_without_exog']['value']
prophet_without_exog_avg = prophet_without_exog_data.mean() if not prophet_without_exog_data.empty else 0

# value_monthly_sarima_with_exog - 실제 컬럼명 사용
monthly_sarima_with_exog_data = df[df['indicator'] == 'value_monthly_sarima_with_exog']['value']
monthly_sarima_with_exog_avg = monthly_sarima_with_exog_data.mean() if not monthly_sarima_with_exog_data.empty else 0

# value_monthly_sarima_without_exog - 실제 컬럼명 사용
monthly_sarima_without_exog_data = df[df['indicator'] == 'value_monthly_sarima_without_exog']['value']
monthly_sarima_without_exog_avg = monthly_sarima_without_exog_data.mean() if not monthly_sarima_without_exog_data.empty else 0

print("\n=== 각 모델별 평균값 ===")
print(f"value_with_exog_평균: {value_with_exog_avg:.6f}")
print(f"value_without_exog_평균: {value_without_exog_avg:.6f}")
print(f"lstm_value_with_exog_평균: {lstm_with_exog_avg:.6f}")
print(f"lstm_value_without_exog_평균: {lstm_without_exog_avg:.6f}")
print(f"prophet_value_with_exog_평균: {prophet_with_exog_avg:.6f}")
print(f"prophet_value_without_exog_평균: {prophet_without_exog_avg:.6f}")
print(f"value_monthly_sarima_with_exog_평균: {monthly_sarima_with_exog_avg:.6f}")
print(f"value_monthly_sarima_without_exog_평균: {monthly_sarima_without_exog_avg:.6f}")

# 각 지표별 데이터 개수도 출력
print("\n=== 각 지표별 데이터 개수 확인 ===")
indicators_to_check = [
    'value_with_exog', 'value_without_exog',
    'lstm_value_with_exog', 'lstm_value_without_exog',
    'prophet_value_with_exog', 'prophet_value_without_exog',
    'value_monthly_sarima_with_exog', 'value_monthly_sarima_without_exog'
]

for indicator in indicators_to_check:
    count = len(df[df['indicator'] == indicator])
    avg_value = df[df['indicator'] == indicator]['value'].mean()
    print(f"{indicator}: {count}개 데이터, 평균값: {avg_value:.2f}")

# 3. mean_market_value 계산 (value_with_exog_평균, value_without_exog_평균 제외)
mean_market_value_list = [
    lstm_with_exog_avg,
    lstm_without_exog_avg,
    prophet_with_exog_avg,
    prophet_without_exog_avg,
    monthly_sarima_with_exog_avg,
    monthly_sarima_without_exog_avg
]

# 0보다 큰 값들만 포함하여 평균 계산
valid_values = [v for v in mean_market_value_list if v > 0]
mean_market_value = np.mean(valid_values) if valid_values else 0

print(f"\n=== 최종 mean_market_value ===")
print(f"mean_market_value 계산에 포함된 지표 개수: {len(valid_values)}")
print(f"mean_market_value: {mean_market_value:.6f}")

# 4. 결과를 딕셔너리로 정리
results = {
    '향후 4분기 예상매출_외생반영': revenue_with_exog_sum,
    '향후 4분기 예상매출_외생미반영': revenue_without_exog_sum,
    'value_with_exog_평균': value_with_exog_avg,
    'value_without_exog_평균': value_without_exog_avg,
    'lstm_value_with_exog_평균': lstm_with_exog_avg,
    'lstm_value_without_exog_평균': lstm_without_exog_avg,
    'prophet_value_with_exog_평균': prophet_with_exog_avg,
    'prophet_value_without_exog_평균': prophet_without_exog_avg,
    'value_monthly_sarima_with_exog_평균': monthly_sarima_with_exog_avg,
    'value_monthly_sarima_without_exog_평균': monthly_sarima_without_exog_avg,
    'mean_market_value': mean_market_value
}

# 5. 결과를 DataFrame으로 변환
result_df = pd.DataFrame([results])

print("\n=== 최종 결과 DataFrame ===")
result_transposed = result_df.T
result_transposed.columns = ['값']
print(result_transposed)

# 6. CSV로 저장 (선택사항)
# result_df.to_csv('forecast_results.csv', index=False, encoding='utf-8-sig')
# print("\n결과가 'forecast_results.csv' 파일로 저장되었습니다.")

=== 데이터 기본 정보 ===
데이터 형태: (176, 9)
컬럼: ['frequency', 'ticker', 'forecast_date', 'indicator', 'value', 'exog_var', 'params', 'target_date', 'valuation_time']

=== 각 indicator별 데이터 개수 ===
indicator
forward_revenue_with_exog            16
forward_revenue_without_exog         16
PSR_monthly_sarima_without_exog      12
PSR_monthly_sarima_with_exog         12
value_monthly_sarima_with_exog       12
value_monthly_sarima_without_exog    12
forecasted_PSR_prophet               12
forecasted_PSR_lstm                  12
lstm_value_with_exog                 12
prophet_value_with_exog              12
lstm_value_without_exog              12
prophet_value_without_exog           12
PSR_quarter_with_exog                 4
revenue_with_exog                     4
PSR_quarter_without_exog              4
revenue_without_exog                  4
value_with_exog                       4
value_without_exog                    4
Name: count, dtype: int64

=== 매출 합산 결과 ===
향후 4분기 예상매출_외생반영: 10458.15
향후 4분기 예상매출_외

In [19]:
# for t in ["AAPL", "MSFT", "GOOGL"]:
#     res = fetch_valuation_by_ticker(db_info, t)
#     if res is not None:
#         print(f"{t}: {len(res)} rows")


In [6]:
# def quick_diagnose(db_info, table_name='US_company_valuation_result'):
#     url = (
#         f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
#         f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
#     )
#     engine = create_engine(url)
#     with engine.connect() as conn:
#         db = conn.execute(text("SELECT DATABASE()")).scalar()
#         ver = conn.execute(text("SELECT VERSION()")).scalar()
#         print(f"📌 Connected DB: {db} | VERSION: {ver}")
#
#         cnt = conn.execute(text("""
#             SELECT COUNT(*) FROM information_schema.tables
#             WHERE table_schema = :s AND table_name = :t
#         """), {"s": db_info["database"], "t": table_name}).scalar()
#         print(f"📦 Table exists? -> {bool(cnt)}")
#
#         like = conn.execute(text(f"SHOW TABLES LIKE :t"), {"t": table_name}).fetchall()
#         print(f"🔎 SHOW TABLES LIKE result: {like}")
#
#         # 권한 점검(에러 무시)
#         try:
#             grants = conn.execute(text("SHOW GRANTS FOR CURRENT_USER()")).fetchall()
#             print("🔐 Grants:")
#             for g in grants:
#                 print("  ", g[0])
#         except Exception as e:
#             print("🔐 Grants check skipped:", e)
#     engine.dispose()

In [9]:
quick_diagnose(db_info)

📌 Connected DB: investar | VERSION: 10.11.6-MariaDB
📦 Table exists? -> False
🔎 SHOW TABLES LIKE result: []
🔐 Grants:
   GRANT ALL PRIVILEGES ON *.* TO `stox7412`@`%` IDENTIFIED BY PASSWORD '*272DDEA3FBB952B113F0D21FD3F15BAF4E29E742' WITH GRANT OPTION
